# 02: OTT Datasets - Cleaning, Transformation & Pipeline Integration

**Project:** Movie and OTT Platform Content Analysis  


### Objectives:
1. Resolve rating/duration shift anomalies.
2. Impute or handle missing values without data loss.
3. Parse release dates into standard temporal types.
4. Standardize age ratings into standardized audience brackets (Kids, Teens, Adults, Unrated).
5. Deconstruct duration into minutes (Movies) and seasons (TV Shows).
6. Integrate datasets via Pandas into `data/processed/unified_ott_dataset.csv`.

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from src.preprocessing import build_unified_dataset, standardize_dataset

# Run the standardized preprocessing pipeline
unified_df = build_unified_dataset(raw_dir='../data/raw', output_path='../data/processed/unified_ott_dataset.csv')
print('Unified Dataset Shape:', unified_df.shape)

Standardizing Netflix from ../data/raw\netflix.csv...


  -> Processed 8,807 records.
Standardizing Amazon Prime from ../data/raw\amazon_prime.csv...
  -> Processed 9,668 records.
Standardizing Disney+ from ../data/raw\disney_plus.csv...


  -> Processed 1,450 records.
Standardizing Hulu from ../data/raw\hulu.csv...
  -> Processed 3,073 records.



Unified dataset created successfully with 22,998 records!
Saved to: ../data/processed/unified_ott_dataset.csv
Unified Dataset Shape: (22998, 24)


## 1. Verifying Cleaned Columns & Data Integrity

In [2]:
print('Cleaned Columns in Unified Dataset:')
for col in unified_df.columns:
    print(f' - {col} ({unified_df[col].dtype})')

Cleaned Columns in Unified Dataset:
 - show_id (str)
 - type (str)
 - title (str)
 - director (str)
 - cast (str)
 - country (str)
 - date_added (str)
 - release_year (Int64)
 - rating (str)
 - duration (str)
 - listed_in (str)
 - description (str)
 - platform (str)
 - platform_code (str)
 - title_id (str)
 - rating_category (str)
 - date_added_clean (datetime64[us])
 - year_added (Int64)
 - month_added (str)
 - duration_int (float64)
 - duration_unit (object)
 - duration_minutes (float64)
 - duration_seasons (float64)
 - primary_genre (str)


## 2. Audience Rating Categorization Distribution

In [3]:
pd.crosstab(unified_df['platform'], unified_df['rating_category'], margins=True)

rating_category,Adults,Kids,Teens,Unrated,All
platform,,,,,
Amazon Prime,2338,1941,4688,701,9668
Disney+,0,765,682,3,1450
Hulu,736,243,1280,814,3073
Netflix,4009,908,3800,90,8807
All,7083,3857,10450,1608,22998


## 3. Duration Parsing Validation

In [4]:
print('Movie Duration in Minutes Summary:')
print(unified_df[unified_df['type'] == 'Movie']['duration_minutes'].describe())

print('\nTV Show Seasons Summary:')
print(unified_df[unified_df['type'] == 'TV Show']['duration_seasons'].describe())

Movie Duration in Minutes Summary:
count    16232.000000
mean        93.566597
std         35.702731
min          0.000000
25%         81.000000
50%         94.000000
75%        109.000000
max        601.000000
Name: duration_minutes, dtype: float64

TV Show Seasons Summary:
count    6579.000000
mean        2.006232
std         2.228398
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        34.000000
Name: duration_seasons, dtype: float64


## 4. Multi-Platform Title Presence

In [5]:
dups = unified_df[unified_df.duplicated(subset=['title', 'type'], keep=False)]
print(f'Total titles available across more than one platform: {dups["title"].nunique():,}')
display(dups.sort_values(by='title')[['title', 'platform', 'type', 'release_year']].head(10))

Total titles available across more than one platform: 720


,title,platform,type,release_year
16280,1 Night in San Diego,Amazon Prime,Movie,2020
20798,1 Night in San Diego,Hulu,Movie,2020
18466,10 Things I Hate About You,Amazon Prime,Movie,1999
19182,10 Things I Hate About You,Disney+,Movie,1999
13346,10.0 Earthquake,Amazon Prime,Movie,2014
22338,10.0 Earthquake,Hulu,Movie,2014
13939,100 Streets,Amazon Prime,Movie,2017
21647,100 Streets,Hulu,Movie,2016
18465,12 Dates of Christmas,Amazon Prime,Movie,2011
19186,12 Dates of Christmas,Disney+,Movie,2011


## 5. Cleaning Pipeline Output
- Cleaned unified dataset saved to `data/processed/unified_ott_dataset.csv`.
- 0 unhandled null values in critical filter dimensions (`platform`, `type`, `rating_category`, `title`).
- Dataset ready for comprehensive Exploratory Data Analysis.